# Building a Dataset from 240 PDF Articles

**Notes**
- The text is saved to **CSV** (no cell-length limit) and to **Excel** with the text split across several columns.
- Ligatures are restored with Unicode **NFKC** normalization, for example the single character `ﬁ` (U+FB01) becomes the two letters `fi`.

**Workflow:**
1. Install and import libraries
2. Set folder locations
3. Per-page text extraction function
4. Text cleaning function
5. Test run on one file
6. Extract all PDFs into a DataFrame
7. Data quality checks
8. Save the dataset (CSV, Excel, Pickle)
9. Verify the output files

> The dataset was built with **pypdf 5.9.0**.

## 1. Install and import libraries

Libraries used:
| Library | Purpose |
|---|---|
| `pypdf` | reads PDFs and extracts text page by page (pure Python, easy to install on Windows) |
| `pandas` | builds and saves the dataset |
| `openpyxl` | writes Excel files (.xlsx) |

In [1]:
%pip install --quiet pypdf==5.9.0 pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import time
import logging
import unicodedata
from pathlib import Path

import pandas as pd
from pypdf import PdfReader

logging.getLogger("pypdf").setLevel(logging.ERROR)

import pypdf
print("pypdf  :", pypdf.__version__)
print("pandas :", pd.__version__)

if pypdf.__version__ != "5.9.0":
    print(f"WARNING: pypdf {pypdf.__version__} is installed, not 5.9.0. The text may differ slightly from the manuscript dataset.")
    print("Run the package installation cell above, then restart the kernel.")

pypdf  : 5.9.0
pandas : 3.0.6


## 2. Folder locations

In [3]:
PDF_FOLDER    = Path(r"C:\Users\marfu\Downloads\journal-20261013T092646Z-1-001\journal")
OUTPUT_FOLDER = Path(r"C:\Users\marfu\Documents\LDA_SBMI")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

KEYWORDS = {
    "s": "Digital servitization",
    "e": "Ethical leadership",
    "d": "Dynamic capabilities",
}

pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))
print("Number of PDFs found:", len(pdf_files))
print("First 5 files:", [p.name for p in pdf_files[:5]])
assert len(pdf_files) == 240, "There should be 240 PDFs. Check PDF_FOLDER."

Number of PDFs found: 240
First 5 files: ['file001d.pdf', 'file001e.pdf', 'file001s.pdf', 'file002d.pdf', 'file002e.pdf']


## 3. Per-page text extraction function

`pypdf` offers two extraction modes:
- **Plain mode** (`extract_text()`): the most accurate for most journal articles, including two-column layouts.
- **Layout mode** (`extract_text(extraction_mode="layout")`): follows the position of characters on the page. This mode helps when justified text causes words to run together without spaces, for example `substantialeffortstoexpand`.

Strategy: use **plain mode**. If a page contains more than 3 very long "words" (22 or more consecutive letters, meaning spaces were lost), try **layout mode** for that page and keep whichever result has fewer long words.

In [4]:
JOINED_WORD_PATTERN = re.compile(r"[A-Za-z]{22,}")   # 22+ letters without a space = likely run-together words

def page_text(page):
    # Extract the text of one PDF page; switch to layout mode if many word spaces are missing.
    text = page.extract_text() or ""
    if len(JOINED_WORD_PATTERN.findall(text)) > 3:
        try:
            layout_text = page.extract_text(extraction_mode="layout") or ""
            if len(JOINED_WORD_PATTERN.findall(layout_text)) < len(JOINED_WORD_PATTERN.findall(text)):
                text = layout_text
        except Exception:
            pass            # if layout mode fails, keep the plain-mode result
    return text


def extract_pdf(pdf_path):
    # Join the text of all pages in one PDF. Returns (raw_text, number_of_pages).
    reader = PdfReader(str(pdf_path))
    all_pages = [page_text(p) for p in reader.pages]
    return "\n".join(all_pages), len(reader.pages)

## 4. Text cleaning function

Steps in `clean_text()`, in order:

| No | Step | Example |
|---|---|---|
| 1 | Unicode **NFKC** normalization: splits ligatures into ordinary letters | `ﬁrms` → `firms`, `eﬀect` → `effect` |
| 2 | Remove *soft hyphens* and invisible (*zero-width*) characters | `https​://` → `https://` |
| 2b | Reattach diacritical marks that became separated from their letter | `Jyv ̈askyl ̈a` → `Jyväskylä` |
| 3 | Rejoin words split by a hyphen at the end of a line | `manu-\nfacturing` → `manufacturing` |
| 4 | Replace control (non-printing) characters with a space | `\x02`, `\x0b` → space |
| 5 | Collapse spaces, tabs and line breaks into a single space | `word   \n  word` → `word word` |

Step 4 is also needed so the text can be written to Excel, because `openpyxl` rejects control characters.

In [5]:
INVISIBLE_PATTERN = re.compile("[\u00ad\u200b\u200c\u200d\u2060\ufeff]")
CONTROL_PATTERN   = re.compile(r"[\x00-\x08\x0b\x0e-\x1f\x7f-\x9f]")
DIACRITIC_PATTERN = re.compile("\\s?([\u0300-\u036f])(\\w)")      # diacritical marks detached from their letter

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)                 # 1. ligatures -> ordinary letters
    text = INVISIBLE_PATTERN.sub("", text)                     # 2. soft hyphens & zero-width characters
    text = DIACRITIC_PATTERN.sub(lambda m: unicodedata.normalize("NFC", m.group(2) + m.group(1)), text)  # 2b. Jyv ̈a -> Jyvä
    text = re.sub(r"(\w)-\n\s*(\w)", r"\1\2", text)            # 3. rejoin hyphenated words
    text = CONTROL_PATTERN.sub(" ", text)                      # 4. control characters
    text = re.sub(r"\s+", " ", text).strip()                   # 5. tidy whitespace
    return text

# Test the function on a small sample
sample = "The \ufb01rms\u2019 e\ufb00ect on manu-\nfacturing   was signi\ufb01cant.\u200b Jyv \u0308askyl \u0308a"
print("Before :", repr(sample))
print("After  :", repr(clean_text(sample)))

Before : 'The ﬁrms’ eﬀect on manu-\nfacturing   was signiﬁcant.\u200b Jyv ̈askyl ̈a'
After  : 'The firms’ effect on manufacturing was significant. Jyväskylä'


## 5. Test run on one file

Before processing all 240 files, check a single file to make sure the result looks reasonable. Look at the number of pages, the number of characters and the opening part of the text.

In [6]:
test_file = pdf_files[0]
raw, n_pages = extract_pdf(test_file)
cleaned = clean_text(raw)

print("File            :", test_file.name)
print("Number of pages :", n_pages)
print("Raw characters  :", f"{len(raw):,}")
print("Clean characters:", f"{len(cleaned):,}")
print("Number of words :", f"{len(cleaned.split()):,}")
print("\nFirst 500 characters:\n", cleaned[:500])

File            : file001d.pdf
Number of pages : 22
Raw characters  : 105,559
Clean characters: 105,187
Number of words : 15,358

First 500 characters:
 International Journal of Management Reviews, V ol. 00, 1–22 (2016) DOI: 10.1111/ijmr.12122 The Essence of Dynamic Capabilities and their Measurement Ola Laaksonen and Mirva Peltoniemi 1 Katajaharjuntie 20, 00200 Helsinki, Finland, and 1Jyväskylä School of Business and Economics, PO Box 35, FI-40014 University of Jyv äskylä, Jyv äskylä, Finland Corresponding author email: mirva.peltoniemi@jyu.fi The growing popularity of explaining firm performance through dynamic capabilities has motivated plent


## 6. Extract all PDFs into a DataFrame

Each PDF produces one row with these columns:

| Column | Content |
|---|---|
| `articlename` | PDF file name |
| `keyword` | search keyword (from the file name suffix) |
| `pages` | number of pages |
| `n_char` | number of characters in the cleaned text |
| `n_words` | number of words (split on spaces) |
| `textdata` | full cleaned text |

In [7]:
rows = []
failed = []
start = time.time()

for i, pdf_path in enumerate(pdf_files, start=1):
    try:
        raw, n_pages = extract_pdf(pdf_path)
        text = clean_text(raw)
        code = pdf_path.stem[-1]                       # last letter of the file name: s / e / d
        rows.append({
            "articlename": pdf_path.name,
            "keyword": KEYWORDS.get(code, "Unknown"),
            "pages": n_pages,
            "n_char": len(text),
            "n_words": len(text.split()),
            "textdata": text,
        })
    except Exception as e:
        failed.append((pdf_path.name, str(e)))
    if i % 20 == 0 or i == len(pdf_files):
        print(f"{i:>3}/{len(pdf_files)} files done  ({time.time() - start:,.0f} seconds)")

df = pd.DataFrame(rows)
print("\nSucceeded :", len(df), "files")
print("Failed    :", len(failed), failed)

 20/240 files done  (35 seconds)
 40/240 files done  (63 seconds)
 60/240 files done  (92 seconds)
 80/240 files done  (126 seconds)
100/240 files done  (186 seconds)
120/240 files done  (211 seconds)
140/240 files done  (236 seconds)
160/240 files done  (272 seconds)
180/240 files done  (317 seconds)
200/240 files done  (371 seconds)
220/240 files done  (440 seconds)
240/240 files done  (481 seconds)

Succeeded : 240 files
Failed    : 0 []


## 7. Data quality checks

Four checks:
1. **Number of articles per keyword.** Should be 80 per group.
2. **Text length statistics.** The `n_char` column.
3. **Empty or very short text.** Scanned PDFs have no text layer and need OCR.
4. **Remaining ligature damage.** Looks for typical broken words such as *dierent* or *benet*; the expected result is 0.

In [8]:
print("1) Number of articles per keyword")
print(df["keyword"].value_counts().to_string(), "\n")

print("2) Text length statistics")
print(df[["pages", "n_char", "n_words"]].describe().round(0).to_string(), "\n")
print("   Texts longer than the Excel cell limit (32,767):", (df["n_char"] > 32767).sum(), "of", len(df), "\n")

df["chars_per_page"] = (df["n_char"] / df["pages"]).round(0)
print("3) PDFs with < 500 characters per page (possibly scanned):")
suspected_scans = df.loc[df["chars_per_page"] < 500, ["articlename", "pages", "n_char"]]
print(suspected_scans.to_string() if len(suspected_scans) else "   none", "\n")

BROKEN_LIGATURE_PATTERN = re.compile(
    r"\b(dierent|benet|benets|signicant|specic|eect|eects|scientic|denition|"
    r"dened|eld|nancial|ecient|dicult|conict|reect|inuence|identied|modied)\b")
df["broken_ligature_words"] = df["textdata"].str.lower().apply(lambda t: len(BROKEN_LIGATURE_PATTERN.findall(t)))
print("4) Total broken ligature words:", df["broken_ligature_words"].sum())
print("   Files with >= 3 broken words:", (df["broken_ligature_words"] >= 3).sum())

1) Number of articles per keyword
keyword
Dynamic capabilities     80
Ethical leadership       80
Digital servitization    80 

2) Text length statistics
       pages    n_char  n_words
count  240.0     240.0    240.0
mean    20.0   85271.0  12331.0
std     10.0   27514.0   4007.0
min      3.0   10790.0   1546.0
25%     14.0   71119.0  10028.0
50%     18.0   84816.0  12132.0
75%     24.0  102119.0  14733.0
max     69.0  201618.0  28361.0 

   Texts longer than the Excel cell limit (32,767): 235 of 240 

3) PDFs with < 500 characters per page (possibly scanned):
   none 

4) Total broken ligature words: 3
   Files with >= 3 broken words: 0


## 8. Saving the dataset

In [9]:
columns = ["articlename", "keyword", "pages", "n_char", "n_words", "textdata"]
keyword_order = {"Digital servitization": 0, "Ethical leadership": 1, "Dynamic capabilities": 2}
dataset = (df[columns]
           .assign(_order=df["keyword"].map(keyword_order))
           .sort_values(["_order", "articlename"])
           .drop(columns="_order")
           .reset_index(drop=True))

# --- CSV (main file) ---
csv_path = OUTPUT_FOLDER / "journal_fulltext.csv"
dataset.to_csv(csv_path, index=False, encoding="utf-8-sig")

# --- Pickle ---
pkl_path = OUTPUT_FOLDER / "journal_fulltext.pkl"
dataset.to_pickle(pkl_path)

# --- Excel: text split into 32,000-character chunks ---
CHUNK_SIZE = 32_000
n_chunks = int(-(-dataset["n_char"].max() // CHUNK_SIZE))   # ceiling division
excel = dataset.drop(columns="textdata").copy()
for k in range(n_chunks):
    excel[f"text_part{k + 1}"] = dataset["textdata"].str.slice(k * CHUNK_SIZE, (k + 1) * CHUNK_SIZE)

xlsx_path = OUTPUT_FOLDER / "journal_fulltext.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    excel.to_excel(writer, sheet_name="dataset", index=False)

for p in (csv_path, xlsx_path, pkl_path):
    print(f"Saved: {p}  ({p.stat().st_size / 1e6:,.1f} MB)")
print("Number of text chunk columns in Excel:", n_chunks)

Saved: C:\Users\marfu\Documents\LDA_SBMI\journal_fulltext.csv  (20.6 MB)
Saved: C:\Users\marfu\Documents\LDA_SBMI\journal_fulltext.xlsx  (6.3 MB)
Saved: C:\Users\marfu\Documents\LDA_SBMI\journal_fulltext.pkl  (20.6 MB)
Number of text chunk columns in Excel: 7


## 9. Verifying the output files

Read all three files back and confirm the text is **identical** to the original DataFrame:
- CSV and Pickle are compared directly.
- Excel: the `text_part1 … text_partN` chunks are joined back together first, then compared.

In [10]:
check_csv = pd.read_csv(csv_path)
check_pkl = pd.read_pickle(pkl_path)
check_xls = pd.read_excel(xlsx_path)
check_xls["textdata"] = check_xls.filter(like="text_part").fillna("").astype(str).agg("".join, axis=1)

original = dataset.set_index("articlename")["textdata"]
for name, d in [("CSV", check_csv), ("Pickle", check_pkl), ("Excel", check_xls)]:
    identical = d.set_index("articlename")["textdata"].reindex(original.index).eq(original).all()
    print(f"{name:<7} → {len(d)} rows, text identical: {identical}")

print("\nLongest text in CSV:", f"{check_csv['textdata'].str.len().max():,}", "characters (not truncated)")

CSV     → 240 rows, text identical: True
Pickle  → 240 rows, text identical: True
Excel   → 240 rows, text identical: True

Longest text in CSV: 201,618 characters (not truncated)
